# Simulating Lensed SN Ia Light Curves with bayesn-td

This tutorial demonstrates how to use bayesn-td to simulate realistic multi-image lensed Type Ia supernova light curves. This is useful for testing the fitting pipeline, exploring observing strategies, or generating training data.

We cover:
1. Loading the model
2. Simulating a single lensed SN with three images
3. Plotting simulated light curves
4. Simulating with fixed vs sampled parameters
5. Simulating in flux vs magnitudes
6. Simulating a small sample and exploring parameter distributions

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from bayesn_td import SEDmodel

model = SEDmodel(load_model='G26x_model')

## 2. Simulating a single lensed SN

Let's simulate a triply-imaged lensed SN Ia at $z = 1.5$, observed in three JWST NIRCam bands. We specify observer-frame observation MJDs and per-image peak MJDs (which encode the time delays). Distance moduli per image encode the magnification differences.

In [ ]:
# Observation MJDs (observer-frame)
mjds = np.arange(59990, 60101, 3.0)

# JWST bands
bands = ['F115W', 'F150W', 'F200W']

# Observer-frame peak MJDs per image (shape (N, n_images))
# Image 0 peaks first, image 1 peaks ~13d later, image 2 peaks ~35d later
peak_mjds = np.array([[60000.0, 60013.0, 60035.0]])

# Distance moduli (encoding magnification differences)
mu = np.array([[44.0, 43.5, 44.2]])  # shape (N, n_images)

# Simulate with photometric noise
data, yerr, params = model.simulate_light_curve_ml(
    mjds=mjds,
    N=1,
    bands=bands,
    peak_mjds=peak_mjds,
    z=1.5,
    yerr=0.05,
    mu=mu,
    ebv_mw=0.01,
)

print(f'Output shape: {data.shape}')
print(f'  (n_mjds x n_bands, n_images, N)')
print(f'\nTrue parameters:')
for key in ['theta', 'AV', 'del_M', 'RV']:
    print(f'  {key}: {params[key]}')

## 3. Plotting simulated light curves

In grid mode (len(mjds) != len(bands)), the simulator observes every MJD in every band. Observations are stored band-major: the first `len(mjds)` rows are `bands[0]`, the next `len(mjds)` rows are `bands[1]`, and so on.

In [ ]:
n_mjds = len(mjds)
n_bands = len(bands)
n_images = peak_mjds.shape[1]
image_labels = ['Image 0', 'Image 1', 'Image 2']
image_colours = ['C0', 'C1', 'C2']

fig, axes = plt.subplots(1, n_bands, figsize=(5 * n_bands, 4), sharey=True)

for b, (ax, band) in enumerate(zip(axes, bands)):
    obs_slice = slice(b * n_mjds, (b + 1) * n_mjds)
    for img in range(n_images):
        ax.errorbar(mjds, data[obs_slice, img, 0],
                    yerr=yerr[obs_slice, img, 0],
                    fmt='o', ms=3, color=image_colours[img],
                    label=image_labels[img], capsize=2)
    ax.set_xlabel('MJD')
    ax.set_title(band)
    ax.invert_yaxis()

axes[0].set_ylabel('Magnitude')
axes[0].legend()
plt.suptitle('Simulated triply-imaged lensed SN Ia', y=1.02)
plt.tight_layout()
plt.show()

## 4. Fixed vs sampled parameters

By default, intrinsic SN parameters ($\theta$, $A_V$, $\delta M$, $\epsilon$) are drawn from the model priors. You can fix any of them to specific values.

In [ ]:
# Simulate with fixed intrinsic parameters
data_fixed, yerr_fixed, params_fixed = model.simulate_light_curve_ml(
    mjds=mjds,
    N=1,
    bands=bands,
    peak_mjds=peak_mjds,
    z=1.5,
    yerr=0.0,  # noiseless
    mu=mu,
    ebv_mw=0.0,
    theta=0.0,    # fix light-curve shape
    AV=0.0,       # fix host dust
    del_M=0.0,    # fix grey offset
    eps=0,         # disable residual colour variation
)

# Plot noiseless light curves
fig, axes = plt.subplots(1, n_bands, figsize=(5 * n_bands, 4), sharey=True)

for b, (ax, band) in enumerate(zip(axes, bands)):
    obs_slice = slice(b * n_mjds, (b + 1) * n_mjds)
    for img in range(n_images):
        ax.plot(mjds, data_fixed[obs_slice, img, 0],
                '-', color=image_colours[img], label=image_labels[img])
    ax.set_xlabel('MJD')
    ax.set_title(band)
    ax.invert_yaxis()

axes[0].set_ylabel('Magnitude')
axes[0].legend()
plt.suptitle('Noiseless light curves (fixed parameters)', y=1.02)
plt.tight_layout()
plt.show()

## 5. Simulating in flux

Set `mag=False` to get fluxes instead of magnitudes.

In [ ]:
data_flux, yerr_flux, params_flux = model.simulate_light_curve_ml(
    mjds=mjds,
    N=1,
    bands=['F150W'],
    peak_mjds=peak_mjds,
    z=1.5,
    yerr=0.03,
    mu=mu,
    ebv_mw=0.01,
    mag=False,
)

fig, ax = plt.subplots(figsize=(6, 4))
for img in range(n_images):
    ax.errorbar(mjds, data_flux[:, img, 0],
                yerr=yerr_flux[:, img, 0],
                fmt='o', ms=3, color=image_colours[img],
                label=image_labels[img], capsize=2)
ax.set_xlabel('MJD')
ax.set_ylabel('Flux (FLUXCAL)')
ax.set_title('Simulated F150W flux')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Simulating a sample and exploring parameter distributions

Let's simulate a batch of 50 lensed SNe and look at the distributions of the sampled intrinsic parameters.

In [ ]:
N_sim = 50
# Per-SN per-image peak MJDs with varying time delays
peak_mjds_batch = np.column_stack([
    np.full(N_sim, 60000.0),
    60000.0 + np.random.uniform(5, 30, N_sim),
    60000.0 + np.random.uniform(15, 60, N_sim),
])  # shape (N_sim, 3)

mu_batch = np.column_stack([
    np.full(N_sim, 44.0),
    np.full(N_sim, 44.0) + np.random.uniform(-1, 0.5, N_sim),
    np.full(N_sim, 44.0) + np.random.uniform(-0.5, 1, N_sim),
])  # shape (N_sim, 3)

data_batch, yerr_batch, params_batch = model.simulate_light_curve_ml(
    mjds=mjds,
    N=N_sim,
    bands=bands,
    peak_mjds=peak_mjds_batch,
    z=np.random.uniform(0.5, 2.5, N_sim),
    yerr=0.05,
    mu=mu_batch,
    ebv_mw=0.01,
)

print(f'Simulated {N_sim} lensed SNe')
print(f'Output data shape: {data_batch.shape}')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))

axes[0].hist(params_batch['theta'], bins=20, alpha=0.7, color='C0')
axes[0].set_xlabel(r'$\theta$')
axes[0].set_ylabel('Count')

axes[1].hist(params_batch['AV'], bins=20, alpha=0.7, color='C1')
axes[1].set_xlabel(r'$A_V$')

axes[2].hist(params_batch['del_M'], bins=20, alpha=0.7, color='C2')
axes[2].set_xlabel(r'$\delta M$')

axes[3].hist(params_batch['RV'], bins=20, alpha=0.7, color='C3')
axes[3].set_xlabel(r'$R_V$')

plt.suptitle('Sampled parameter distributions (N=50)', y=1.02)
plt.tight_layout()
plt.show()